# SolarCast - NOAA HRRR Forecast Weather Extraction (Google Colab)

This notebook executes the Mode F1 HRRR forecast feature extraction pipeline on Google Colab.
Cloud infrastructure provides ~50-100x higher throughput and lower latency to AWS S3 us-west-1
compared to local residential connections.

### Setup Instructions:
1. Upload the folder `SolarCast_inputs` (from `D:\SolarCast_inputs` on your laptop) to your Google Drive root folder:
   `My Drive / SolarCast_inputs /` containing:
   - `hourly.parquet`
   - `hrrr_coverage_scan.parquet`
2. Run the cells in order.
3. Extracted monthly parts are saved directly to Google Drive at `My Drive / SolarCast_parts/` so progress is persisted across disconnections.


In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# Cell 2: Clone repository and checkout branch
import os
import sys

if not os.path.exists("/content/SolarCast"):
    os.system("git clone https://github.com/AwanTech095/SolarCast.git /content/SolarCast")
os.chdir("/content/SolarCast")
os.system("git checkout m1/hrrr-forecast-features")
sys.path.insert(0, "/content/SolarCast")


In [ ]:
# Cell 3: Install extraction dependencies
import os

os.system("pip install -q -r requirements-extract.txt")


In [ ]:
# Cell 4: Copy input files from Google Drive to local repository
import os
import shutil
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)
drive_input = Path("/content/drive/MyDrive/SolarCast_inputs")
for fname in ["hourly.parquet", "hrrr_coverage_scan.parquet"]:
    src = drive_input / fname
    dst = Path("data/processed") / fname
    if src.exists():
        shutil.copy2(src, dst)
        print(f"Copied {fname} ({dst.stat().st_size:,} bytes)")
    else:
        print(f"Warning: {src} not found on Drive. Please upload D:\\SolarCast_inputs.")


In [ ]:
# Cell 5: Self-test - extract 2022-06-13 00:00 UTC and verify against reference
import numpy as np
import pandas as pd

from src.data.hrrr import (
    HRRRExtractorConfig,
    extract_hour_weather,
    get_thread_fs,
    load_coverage_scan_set,
)

cfg = HRRRExtractorConfig(variables=["tmp", "tcdc"])
fs = get_thread_fs()
scan_set = load_coverage_scan_set("data/processed/hrrr_coverage_scan.parquet")

t = pd.Timestamp("2022-06-13 00:00:00", tz="UTC")
res = extract_hour_weather(t, cfg, fs, scan_set)

print("Self-test extraction result:")
for k, v in res.items():
    print(f"  {k}: {v}")

ref_tmp_c = 25.6
assert np.isclose(res["tmp_c"], ref_tmp_c, atol=0.1), f"Expected {ref_tmp_c} C, got {res['tmp_c']}"
assert res["extraction_status"].startswith(("SUCCESS", "FALLBACK_SUCCESS"))
print("\n[PASSED] Self-test successfully verified against reference value!")


In [ ]:
# Cell 6: Run extraction with configurable start/end dates
# Priority 1: 2022-10-17 to 2023-11-09 (Test & latest Dev fold)
# Priority 2: 2021-10-17 to 2022-10-16 (Prior training fold)
# Priority 3: Older months as time permits
import os

start_date = "2022-10-17"
end_date = "2023-11-09"
variables = "tmp,tcdc"
threads = 16
parts_dir = "/content/drive/MyDrive/SolarCast_parts"

cmd = (
    f"python scripts/hrrr_extract.py "
    f"--start {start_date} "
    f"--end {end_date} "
    f"--variables {variables} "
    f"--threads {threads} "
    f"--parts-dir {parts_dir}"
)
print(f"Executing: {cmd}")
os.system(cmd)


In [ ]:
# Cell 7: Check live progress and inspect completed parts on Google Drive
from pathlib import Path

log_file = Path("logs/hrrr_extract.log")
if log_file.exists():
    lines = log_file.read_text(encoding="utf-8").strip().splitlines()
    print("\n".join(lines[-15:]))

parts_path = Path("/content/drive/MyDrive/SolarCast_parts")
if parts_path.exists():
    parts = list(parts_path.glob("*.parquet"))
    print(f"\nCompleted parts on Drive: {len(parts)}")
    for p in sorted(parts):
        print(f"  {p.name} ({p.stat().st_size / 1024:.1f} KB)")


In [ ]:
# Cell 8: Merge completed parts into final consolidated dataset on Drive
import os

import pandas as pd

parts_dir = "/content/drive/MyDrive/SolarCast_parts"
merged_file = "/content/drive/MyDrive/SolarCast_parts/hrrr_forecast_features.parquet"

cmd = f"python scripts/hrrr_extract.py --parts-dir {parts_dir} --merged-file {merged_file} --merge"
os.system(cmd)

if os.path.exists(merged_file):
    df = pd.read_parquet(merged_file)
    print(f"Successfully merged: {len(df):,} records")
    print(df.info())
    print(df.head())
